# General aim
This notebook was developed as part of the Master’s thesis of June Rossen and is the fourth of five notebooks used to prepare data for a wind power placement multi-objective optimization.

This fourth notebook focuses on normalizing the optimization criteria to have a final geodataframe, ready for optimization, with values included between 0 and 1. The optimization will try to minimize total impacts/costs, so values close to 0 should be understood as "good" and values close to 1 as "bad".

Based on the nature of the criteria, they are either kept as is (or divided by two), linearly normalized using a max-min normalization, or log transformed before max-min normalization. Sometimes, excessive values are cut-off and put to 0 or 1.

In a subsequent step, the criteria are merged into several upper level categories, two for each dimension (techno-economic, social, environmental).

## Linear normalization
Most of the criteria are linearly normalized. This includes:
- LCOE  (with values above 60 oere/kWh considered as "bad", so with a value of 1)
- Distance to grid
- Distance to transport infrastructure
- Icing issues
-
- Minimum distance to people
-
- Infrastructure density
- Intersection with sensitive fauna (birds)

## Logarithmic transformation
Criteria linked to the number of people impacted are logarithmicaly transformed. This was decided because while impacting 1'000'000 people can be understood as "really bad", impacting 100'000 people is still "very bad", and counting it as "10x better" does not seem right. If population distribution across the country was more uniform, then linear normalization could have been interesting. The criteria concerned are:
- Viewshed study (in a 55km radius)
- Sound impact study (in a 5km radius)

## Binary transformation
Some criteria are already yes/no, i.e. with 0/1 values. This step goes through them to ensure that they have 0 = good, 1 = bad. Those are
- Intersection with recreational areas
- Intersection with semi-domesticated reindeer areas
-
- Presence of hydropower
- Presence of wind turbines

## Keep as is
Some criteria do not need changing, because they are already between 0 and 1, with closer to 0 being good and closer to 1 being bad. Those include:
- Coverage of wetlands
- Coverage of forests       (though it is divided by 2, considering it is not "as valuable" as wetlands)
- Coverage of old forests   (though it is divided by 2, considering this "half score" is added to the "half score" of forests in general (which should overlap with old forests), to give in the end a full score with the same weight as the wetlands' score)

## Criteria merging
For the criteria merging, these are simply summed up and divided by the number of criteria per category, in order to have a maximum value of 1 per category. We have:
- Techno-economic dimension:
    - Costs (LCOE)
    - Technical difficulty (distance to grid, distance to transport infrastructure, icing issues, previous presence of wind turbine, previous presence of hydropower)
- Social dimension:
    - Daily exposure (viewshed study, sound study, distance to people)
    - Cultural (recreational areas, semi-domesticated reindeer areas)
- Environmental dimension:
    - Landscape change (wetlands, forests, old forests, infrastructure density)
    - Sensitive fauna (bird impact potential)

## Notebook workflow
- First, the necessary librairies are imported and the data is loaded.
- Then, a function is defined to do the different normalizations.
- The normalizations are applied.
- Merging is done.
- Results are saved.
- An extra aggregation procedure is done over 2 and 5km scale, in order to reduce the total number of cells kept.

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np

In [2]:
# Load vector and csv data
NO_grid = gpd.read_file("data/SSB_techlegalfiltered_opticriteria.gpkg")

In [3]:
NO_grid_normalized = NO_grid[["SSBid", "Komm2016", "Fylk2016", "geometry"]].copy()

In [4]:
NO_grid.head()

,SSBid,Komm2016,Fylk2016,pct_NoGo,avg_windspeed,avg_AEP_3MW_1turbine_kWh,avg_AEP_5MW_1turbine_kWh,potential_nb_3MW_turbines,potential_nb_5MW_turbines,avg_AEP_3MW,...,population_in_5km,dist_to_individuals,recreational_area,domesticated_reindeer,wetlands,forest,old_forest,infrastructure_index,bird_impact_potential,geometry
0,22990006544000,0101,01,1.125000e-02,7.133292,11259643.0,18712946.0,4.0,3.0,45.038572,...,458,1000.0,1,0,0.0000,0.988750,0.164096,2.672269,0.222749,"MULTIPOLYGON (((4408499.644 3988462.997, 44075..."
1,22990006545000,0101,01,0.000000e+00,7.148756,11783823.0,19610206.0,4.0,3.0,47.135292,...,472,1000.0,1,0,0.0000,0.999375,0.200320,2.571814,0.219633,"MULTIPOLYGON (((4408425.191 3989458.236, 44074..."
2,22990006550000,0101,01,1.937500e-02,7.336300,11115102.0,18448636.0,4.0,2.0,44.460408,...,776,1000.0,0,0,0.2975,0.682500,0.041600,3.707362,0.186828,"MULTIPOLYGON (((4408052.815 3994434.348, 44070..."
3,22980006555000,0101,01,1.110223e-16,7.359521,11519288.0,19160622.0,4.0,3.0,46.077152,...,4709,1000.0,1,0,0.0000,0.966875,0.316928,3.491421,0.171705,"MULTIPOLYGON (((4406681.094 3999336.954, 44056..."
4,22990006555000,0101,01,1.110223e-16,7.435547,12323313.0,20524556.0,4.0,3.0,49.293252,...,1609,2000.0,1,0,0.0150,0.985000,0.357888,3.337394,0.170284,"MULTIPOLYGON (((4407680.258 3999410.323, 44066..."


In [5]:
def normalize(criteria, log=False, more_better=False, max_val=None, min_val=None):
    """
    Normalize a Pandas series (criteria) to [0, 1], 0 being "good" and 1 being "bad".
    log=True log transforms the data before normalizing it.
    more_better=True reverses the scale so that high values -> low normalized scores, to fit the optimization direction.
    max_val and min_val can be used to restrict the normalization range.
    """
    # set max and min values to optionaly predefined ones, otherwise simply take the max/min of the criteria series
    if max_val:
        maximum = max_val
    else:
        maximum = criteria.max()
    if min_val:
        minimum = min_val
    else:
        minimum = criteria.min()

    if log:
        # replace zeros by NaNs before log-transform (avoids -infinity), and refill the NaNs after with 0
        criteria = np.log10(criteria.replace(0, np.nan)).fillna(0)

        # in case max_val or min_val were given, also log transform them (if they are not <= 0)
        if maximum > 0:
            maximum = np.log10(maximum)
        if minimum > 0:
            minimum = np.log10(minimum)
    
    # if divide-by-zero, return -1
    if maximum == minimum:
        return pd.Series(-1, index=criteria.index)

    normalized = ((criteria - minimum) / (maximum - minimum)).clip(0, 1)    # clip is useful for when predefined max/min values were given
    
    # flip the scale if needed
    if more_better:
        normalized = 1 - normalized
     
    return normalized


## Normalize techno-economic criteria

In [6]:
# keep as is
NO_grid_normalized["AEP_GWh"] = NO_grid["AEP_GWh"]

# above 60 oere/kWh, assume it is not competitve enough against hydropower, the major source of renewable energy in Norway
NO_grid_normalized["LCOE"] = normalize(NO_grid["avg_LCOE"], max_val=60)

# simple linear normalization
NO_grid_normalized["Distance_to_grid"] = normalize(NO_grid["dist_to_grid"])
NO_grid_normalized["Distance_to_road"] = normalize(NO_grid["dist_to_road"])
NO_grid_normalized["Icing_potential"] = normalize(NO_grid["icing_potential"])

# if there is one turbine (hydropower plant) or more, it is good, assign 0. Else assign 1
NO_grid_normalized["Existing_turbines"] = 1 - NO_grid["nb_existing_turbines"].clip(upper=1)
NO_grid_normalized["Existing_hydropower"] = 1 - NO_grid["nb_existing_hydro_plants"].clip(upper=1)


## Normalize social criteria

In [7]:
# log transform to take into account the non-uniform distribution of the population across the country
NO_grid_normalized["Visible_population"] = normalize(NO_grid["visible_pop"], log=True)
NO_grid_normalized["Population_in_5km"] = normalize(NO_grid["population_in_5km"], log=True)
NO_grid_normalized["Distance_to_individuals"] = normalize(NO_grid["dist_to_individuals"], log=True, more_better=True)

# keep as it
NO_grid_normalized["Recreational_area"] = NO_grid["recreational_area"]
NO_grid_normalized["Semidomesticated_reindeer"] = NO_grid["domesticated_reindeer"]

## Normalize environmental criteria

In [8]:
# keep as it
NO_grid_normalized["Wetland"] = NO_grid["wetlands"]

# divide by 2
NO_grid_normalized["Forest"] = NO_grid["forest"] / 2
NO_grid_normalized["Old_forest"] = NO_grid["old_forest"] / 2

# linear normalization, but invert (we want to use maximal infrastructure density)
NO_grid_normalized["Infrastructure_index"] = normalize(NO_grid["infrastructure_index"], more_better=True)

# linear normalization
NO_grid_normalized["Bird_impact_potential"] = normalize(NO_grid["bird_impact_potential"])

In [9]:
NO_grid_normalized

,SSBid,Komm2016,Fylk2016,geometry,AEP_GWh,LCOE,Distance_to_grid,Distance_to_road,Icing_potential,Existing_turbines,...,Visible_population,Population_in_5km,Distance_to_individuals,Recreational_area,Semidomesticated_reindeer,Wetland,Forest,Old_forest,Infrastructure_index,Bird_impact_potential
0,22990006544000,0101,01,"MULTIPOLYGON (((4408499.644 3988462.997, 44075...",56.138838,0.323949,0.004853,0.012004,0.012781,1,...,0.747862,0.548141,1.000000,1,0,0.000000,0.494375,0.082048,0.794072,0.294641
1,22990006545000,0101,01,"MULTIPOLYGON (((4408425.191 3989458.236, 44074...",58.830618,0.321255,0.015414,0.018520,0.011914,1,...,0.749495,0.550834,1.000000,1,0,0.000000,0.499688,0.100160,0.801813,0.290500
2,22990006550000,0101,01,"MULTIPOLYGON (((4408052.815 3994434.348, 44070...",44.460408,0.280815,0.004971,0.011310,0.009663,1,...,0.752106,0.595314,1.000000,0,0,0.297500,0.341250,0.020800,0.714307,0.246901
3,22980006555000,0101,01,"MULTIPOLYGON (((4406681.094 3999336.954, 44056...",57.481866,0.279645,0.006413,0.030897,0.007839,1,...,0.755572,0.756626,1.000000,1,0,0.000000,0.483437,0.158464,0.730947,0.226802
4,22990006555000,0101,01,"MULTIPOLYGON (((4407680.258 3999410.323, 44066...",61.573668,0.265917,0.028070,0.048556,0.013924,1,...,0.754772,0.660553,0.819540,1,0,0.015000,0.492500,0.178944,0.742817,0.224913
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134278,31040007818000,2030,20,"MULTIPOLYGON (((5120420.194 5303057.846, 51194...",34.268688,0.107681,0.012909,0.025474,0.071891,1,...,0.000000,0.000000,0.324012,0,1,0.000000,0.095000,0.003392,0.959859,0.181380
134279,31050007818000,2030,20,"MULTIPOLYGON (((5121425.654 5303108.24, 512042...",36.018080,0.127799,0.008674,0.008330,0.070802,1,...,0.000000,0.000000,0.307082,1,1,0.000000,0.070312,0.000000,0.863041,0.138479
134280,31020007819000,2030,20,"MULTIPOLYGON (((5118336.321 5303939.79, 511733...",55.297680,0.118537,0.019633,0.033357,0.057456,1,...,0.000000,0.000000,0.348616,0,1,0.000000,0.075000,0.008640,0.982919,0.362546
134281,31030007819000,2030,20,"MULTIPOLYGON (((5119341.831 5303990.254, 51183...",61.319484,0.115811,0.009592,0.022581,0.070691,1,...,0.000000,0.000000,0.331452,1,1,0.038125,0.008125,0.000000,0.910888,0.197009


In [10]:
NO_grid_normalized.to_file("data/SSB_techlegalfiltered_opticriteria_normalized.gpkg", driver="GPKG")

## Merge the criteria into the 6 categories

In [11]:
NO_grid_normalized = gpd.read_file("data/SSB_techlegalfiltered_opticriteria_normalized.gpkg")

In [12]:
NO_grid_merged = NO_grid_normalized[["SSBid", "Komm2016", "Fylk2016", "AEP_GWh", "geometry"]].copy()

In [13]:
NO_grid_merged["Daily_exposure"] = (NO_grid_normalized["Visible_population"] + NO_grid_normalized["Population_in_5km"] + NO_grid_normalized["Distance_to_individuals"]) / 3
NO_grid_merged["Culture"] = (NO_grid_normalized["Recreational_area"] + NO_grid_normalized["Semidomesticated_reindeer"]) / 2

NO_grid_merged["Landscape_change"] = (NO_grid_normalized["Wetland"] + NO_grid_normalized["Forest"] + NO_grid_normalized["Old_forest"] + NO_grid_normalized["Infrastructure_index"]) / 2
NO_grid_merged["Sensitive_fauna"] = NO_grid_normalized["Bird_impact_potential"]

NO_grid_merged["Technical_difficulty"] = (NO_grid_normalized["Distance_to_grid"] + NO_grid_normalized["Distance_to_road"] + NO_grid_normalized["Icing_potential"] + NO_grid_normalized["Existing_turbines"] + NO_grid_normalized["Existing_hydropower"]) / 5
NO_grid_merged["Costs"] = NO_grid_normalized["LCOE"]

In [14]:
NO_grid_merged

,SSBid,Komm2016,Fylk2016,AEP_GWh,geometry,Daily_exposure,Culture,Landscape_change,Sensitive_fauna,Technical_difficulty,Costs
0,22990006544000,0101,01,56.138838,"MULTIPOLYGON (((4408499.644 3988462.997, 44075...",0.765334,0.5,0.685248,0.294641,0.405928,0.323949
1,22990006545000,0101,01,58.830618,"MULTIPOLYGON (((4408425.191 3989458.236, 44074...",0.766777,0.5,0.700830,0.290500,0.409170,0.321255
2,22990006550000,0101,01,44.460408,"MULTIPOLYGON (((4408052.815 3994434.348, 44070...",0.782473,0.0,0.686928,0.246901,0.405189,0.280815
3,22980006555000,0101,01,57.481866,"MULTIPOLYGON (((4406681.094 3999336.954, 44056...",0.837400,0.5,0.686424,0.226802,0.409030,0.279645
4,22990006555000,0101,01,61.573668,"MULTIPOLYGON (((4407680.258 3999410.323, 44066...",0.744955,0.5,0.714630,0.224913,0.418110,0.265917
...,...,...,...,...,...,...,...,...,...,...,...
134278,31040007818000,2030,20,34.268688,"MULTIPOLYGON (((5120420.194 5303057.846, 51194...",0.108004,0.5,0.529125,0.181380,0.422055,0.107681
134279,31050007818000,2030,20,36.018080,"MULTIPOLYGON (((5121425.654 5303108.24, 512042...",0.102361,1.0,0.466677,0.138479,0.417561,0.127799
134280,31020007819000,2030,20,55.297680,"MULTIPOLYGON (((5118336.321 5303939.79, 511733...",0.116205,0.5,0.533279,0.362546,0.422089,0.118537
134281,31030007819000,2030,20,61.319484,"MULTIPOLYGON (((5119341.831 5303990.254, 51183...",0.110484,1.0,0.478569,0.197009,0.420573,0.115811


In [15]:
NO_grid_merged.to_file("data/SSB_techlegalfiltered_opticriteria_normalized_merged.gpkg", driver="GPKG")

## Get an overview of the 6 impact categories

In [16]:
NO_grid_merged["Daily_exposure"].agg(["sum", "mean", "min", "max", "std"])


sum     56432.609091
mean        0.420251
min         0.000000
max         0.995099
std         0.203737
Name: Daily_exposure, dtype: float64

In [17]:
NO_grid_merged["Culture"].agg(["sum", "mean", "min", "max", "std"])


sum     84613.500000
mean        0.630113
min         0.000000
max         1.000000
std         0.363507
Name: Culture, dtype: float64

In [18]:
NO_grid_merged["Landscape_change"].agg(["sum", "mean", "min", "max", "std"])


sum     84627.607845
mean        0.630218
min         0.043520
max         1.073667
std         0.142842
Name: Landscape_change, dtype: float64

In [19]:
NO_grid_merged["Sensitive_fauna"].agg(["sum", "mean", "min", "max", "std"])


sum     18264.478516
mean        0.136015
min         0.000000
max         1.000000
std         0.090100
Name: Sensitive_fauna, dtype: float32

In [20]:
NO_grid_merged["Technical_difficulty"].agg(["sum", "mean", "min", "max", "std"])


sum     61913.679486
mean        0.461069
min         0.014482
max         0.790549
std         0.054078
Name: Technical_difficulty, dtype: float64

In [21]:
NO_grid_merged["Costs"].agg(["sum", "mean", "min", "max", "std"])


sum     53343.159933
mean        0.397244
min         0.000000
max         1.000000
std         0.254582
Name: Costs, dtype: float64

## Aggregating the results over a 2x2 and a 5x5km grid, in order to reduce the total number of cells 

In [31]:
NO_grid_2km = gpd.read_file("data/SSB_2km.shp")
NO_grid_5km = gpd.read_file("data/SSB_5km.gpkg")

In [32]:
NO_grid_merged["centroid"] = NO_grid_merged.geometry.centroid
NO_grid_merged.set_geometry("centroid", inplace=True)

In [33]:
NO_grid_2km.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 120504 entries, 0 to 120503
Data columns (total 12 columns):
 #   Column      Non-Null Count   Dtype   
---  ------      --------------   -----   
 0   Join_Count  120504 non-null  int64   
 1   TARGET_FID  120504 non-null  int64   
 2   SSBid       120504 non-null  object  
 3   ost         120504 non-null  int64   
 4   nord        120504 non-null  int64   
 5   CENTROID_X  120504 non-null  float64 
 6   CENTROID_Y  120504 non-null  float64 
 7   Shape_Leng  120504 non-null  float64 
 8   Shape_Area  120504 non-null  float64 
 9   Komm2016    120504 non-null  object  
 10  Fylk2016    120504 non-null  object  
 11  geometry    120504 non-null  geometry
dtypes: float64(4), geometry(1), int64(4), object(3)
memory usage: 11.0+ MB


In [34]:
NO_grid_5km.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 20747 entries, 0 to 20746
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   geometry_bbox.xmin  20747 non-null  float64 
 1   geometry_bbox.ymin  20747 non-null  float64 
 2   geometry_bbox.xmax  20747 non-null  float64 
 3   geometry_bbox.ymax  20747 non-null  float64 
 4   geometry            20747 non-null  geometry
dtypes: float64(4), geometry(1)
memory usage: 810.6 KB


In [35]:
NO_grid_2km["SSBid_2km"] = NO_grid_2km["SSBid"]
NO_grid_5km["id"] = NO_grid_5km.index

In [36]:
NO_grid_5km = NO_grid_5km.to_crs(NO_grid_merged.crs)
NO_grid_2km = NO_grid_2km.to_crs(NO_grid_merged.crs)

In [37]:
# Spatial join: assign each 1km cell to its containing 2km or 5km cell
joined_2km = gpd.sjoin(NO_grid_merged, NO_grid_2km[["SSBid_2km", "geometry"]], how="left", predicate="within")
joined_5km = gpd.sjoin(NO_grid_merged, NO_grid_5km[["id", "geometry"]], how="left", predicate="within")

In [38]:
attributes = ["AEP_GWh", "Landscape_change", "Sensitive_fauna", "Daily_exposure", "Culture", "Technical_difficulty", "Costs"]

agg_2km = joined_2km.groupby("SSBid_2km")[attributes].sum().reset_index()
agg_5km = joined_5km.groupby("id")[attributes].sum().reset_index()

NO_grid_merged_2km = NO_grid_2km.merge(agg_2km, on="SSBid_2km", how="left")
NO_grid_merged_5km = NO_grid_5km.merge(agg_5km, on="id", how="left")

NO_grid_merged_2km_noNaNs = NO_grid_merged_2km.dropna()
NO_grid_merged_5km_noNaNs = NO_grid_merged_5km.dropna()


In [39]:
NO_grid_merged_2km_noNaNs.to_file("data/SSB_2km_techlegalfiltered_opticriteria_normalized_merged.gpkg", driver="GPKG")
NO_grid_merged_5km_noNaNs.to_file("data/SSB_5km_techlegalfiltered_opticriteria_normalized_merged.gpkg", driver="GPKG")